# Inception V3 XAI on clean checkpoint (inception_v3_seed7)

299px input, ImageNet normalisation, Grad-CAM on `Mixed_7c`, aux logits disabled for inference. Frozen 30-image set (upsampled to 299), deletion/insertion + heatmap saving.

**Attach:** dataset, 03a_cnns_seeded, 06_build_xai_imageset. GPU on.

In [1]:
import torch, torch.nn as nn, numpy as np
from torchvision import datasets, transforms, models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [2]:
# Dataset at 299px + ImageNet norm (SHAP background is drawn from this)
transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
dataset = datasets.ImageFolder(
    root='/kaggle/input/datasets/shajinrp/diabetic-retinopathy/Dataset',
    transform=transform)
print('Classes:', dataset.classes, '| size:', len(dataset))

Classes: ['Mild', 'Moderate', 'No_DR', 'Proliferate_DR', 'Severe'] | size: 3554


In [3]:
# Build Inception V3 exactly as trained in 03a, then load clean weights
inception = models.inception_v3(weights=None, aux_logits=True, init_weights=False)
inception.fc = nn.Linear(inception.fc.in_features, 5)
inception.AuxLogits.fc = nn.Linear(inception.AuxLogits.fc.in_features, 5)
state = torch.load('/kaggle/input/notebooks/tochyokafor/03a-cnns-seeded/inception_v3_seed7.pth',
                   map_location=device)
inception.load_state_dict(state)
inception.aux_logits = False          # single-output for clean inference/XAI
inception.AuxLogits = None
inception = inception.to(device).eval()
model = inception                     # alias so shared code/Block2 works
print('Loaded inception_v3_seed7.pth (aux disabled for inference)')

Loaded inception_v3_seed7.pth (aux disabled for inference)


In [4]:
!pip install shap lime -q

## XAI method definitions (same fixed versions as the CNN notebook)

In [5]:
# ============================================================
#  CNN EXPLAINABILITY PIPELINE — COMPLETE FINAL VERSION
#  Includes: Grad-CAM, Saliency, SHAP, LIME
#
#  Plot A — Grad-CAM Grid      (Original | Heatmap | Overlay)
#  Plot B — Correct vs Wrong   (Original | Overlay side-by-side)
#  Plot C — Mean CAM Intensity per predicted class
#  Plot D — Softmax Confidence grid
#  Plot E — Saliency Map Grid  (Original | Saliency | Overlay)
#  Plot F — SHAP Grid          (Original | Intensity | Signed | Overlay)
#  Plot G — LIME Grid          (Original | Segments  | Mask Overlay)
#  Plot H — Full Comparison    (Original | Grad-CAM | Saliency | SHAP | LIME)
#  Plot I — Feature Maps       (first image, conv4 channels)
#
#  ── Run this in a separate Kaggle cell FIRST ──
#  !pip install shap lime -q
#
#  Paste this AFTER your existing training/test cells.
#  Requires: model, device, dataset  (already defined)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter
import torch
import torch.nn as nn
import torch.nn.functional as F
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# ── Colour palette for 5 DR severity classes ─────────────────
CLASS_COLOURS = ["#4CAF50", "#8BC34A", "#FFC107", "#FF5722", "#F44336"]


# ═══════════════════════════════════════════════════════════════
# 0.  PATCH inplace ReLUs (prevents silent gradient bugs)
# ═══════════════════════════════════════════════════════════════
for module in model.modules():
    if isinstance(module, nn.ReLU):
        module.inplace = False


# ═══════════════════════════════════════════════════════════════
# 1.  HELPERS
# ═══════════════════════════════════════════════════════════════
def denorm(tensor_batch):
    """
    (N,C,H,W) tensor in [0,1] → (N,H,W,3) float numpy in [0,1].
    Uncomment MEAN/STD lines if you add ImageNet normalisation.
    """
    arr = tensor_batch.detach().cpu().permute(0, 2, 3, 1).numpy()
    MEAN = np.array([0.485, 0.456, 0.406])
    STD  = np.array([0.229, 0.224, 0.225])
    arr  = arr * STD + MEAN
    return np.clip(arr, 0, 1)


def get_heatmap(mask, colormap="jet"):
    """Convert a [0,1] mask → (H,W,3) RGB heatmap."""
    return cm.get_cmap(colormap)(mask)[..., :3]


def get_overlay(img_np, mask, colormap="jet", alpha=0.55):
    """Blend a [0,1] mask onto a [0,1] RGB image."""
    heatmap = get_heatmap(mask, colormap)
    return np.clip((1 - alpha) * img_np + alpha * heatmap, 0, 1)


def sample_dataset(dataset, n, seed=42):
    """Reproducibly sample n indices from a dataset."""
    g      = torch.Generator().manual_seed(seed)
    idx    = torch.randperm(len(dataset), generator=g)[:n].tolist()
    imgs   = torch.stack([dataset[i][0] for i in idx])
    labels = [dataset[i][1] for i in idx]
    return imgs, labels, idx


# ═══════════════════════════════════════════════════════════════
# 2.  GRAD-CAM  (batch-aware)
#     Fix: percentile clipping (5–95%) before normalisation
#          prevents washed-out low-contrast heatmaps
# ═══════════════════════════════════════════════════════════════
class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.activations  = None
        self.gradients    = None
        self._register_hooks()

    def _register_hooks(self):
        def fwd(module, inp, out):
            self.activations = out.detach()

        def bwd(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self.target_layer.register_forward_hook(fwd)
        self.target_layer.register_full_backward_hook(bwd)

    def generate(self, input_tensor, target_class=None):
        """
        Returns
        -------
        cams     : (N, H, W) numpy in [0,1]
        pred_ids : (N,)      numpy int
        logits   : (N, C)    numpy float
        """
        self.model.eval()
        inp    = input_tensor.requires_grad_(True)
        out    = self.model(inp)
        logits = out if isinstance(out, torch.Tensor) else out[0]

        pred_ids = logits.argmax(dim=1)
        targets  = (pred_ids if target_class is None
                    else torch.full_like(pred_ids, target_class))

        self.model.zero_grad()
        one_hot = torch.zeros_like(logits)
        one_hot.scatter_(1, targets.unsqueeze(1), 1.0)
        logits.backward(gradient=one_hot, retain_graph=False)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = F.relu((weights * self.activations).sum(dim=1))

        H, W = input_tensor.shape[2], input_tensor.shape[3]
        cams = []
        for c in cam:
            c = F.interpolate(c.unsqueeze(0).unsqueeze(0),
                              size=(H, W), mode="bilinear",
                              align_corners=False)
            c = c.squeeze().cpu().numpy()
            p5, p95 = np.percentile(c, 5), np.percentile(c, 95)
            c = np.clip(c, p5, p95)
            mn, mx = c.min(), c.max()
            cams.append((c - mn) / (mx - mn + 1e-8))

        return (np.stack(cams),
                pred_ids.detach().cpu().numpy(),
                logits.detach().cpu().numpy())


# ═══════════════════════════════════════════════════════════════
# 3.  SALIENCY MAP  (batch)
#     Fixes: detach() leaf tensor, Gaussian smoothing,
#            percentile normalisation (1–99%)
# ═══════════════════════════════════════════════════════════════
def compute_saliency_batch(model, input_tensor):
    """Returns (N, H, W) saliency maps in [0,1]."""
    model.eval()
    inp    = input_tensor.clone().detach().requires_grad_(True)
    logits = model(inp)

    model.zero_grad()
    pred_ids = logits.argmax(dim=1)
    one_hot  = torch.zeros_like(logits)
    one_hot.scatter_(1, pred_ids.unsqueeze(1), 1.0)
    logits.backward(gradient=one_hot)

    sal    = inp.grad.data.abs()
    sal, _ = sal.max(dim=1)
    sal    = sal.cpu().numpy()

    out_sal = []
    for s in sal:
        s = gaussian_filter(s, sigma=2)
        p1, p99 = np.percentile(s, 1), np.percentile(s, 99)
        s = np.clip(s, p1, p99)
        s = (s - s.min()) / (s.max() - s.min() + 1e-8)
        out_sal.append(s)
    return np.stack(out_sal)


# ═══════════════════════════════════════════════════════════════
# 4.  SHAP — GradientExplainer
#     Uses 50 background images as baseline reference.
#     Returns unsigned intensity map + signed red/blue map.
# ═══════════════════════════════════════════════════════════════
def compute_shap(model, exp_imgs, device, dataset,
                 n_background=50, seed=42):
    """
    Returns shap_maps (N,H,W) in [0,1], shap_rgb (N,H,W,3), pred_ids (N,).
    Robust to SHAP version differences in output layout.
    """
    model.eval()
    g      = torch.Generator().manual_seed(seed + 1)
    bg_idx = torch.randperm(len(dataset), generator=g)[:n_background].tolist()
    bg     = torch.stack([dataset[i][0] for i in bg_idx]).to(device)

    explainer   = shap.GradientExplainer(model, bg)
    shap_values = explainer.shap_values(exp_imgs)

    with torch.no_grad():
        logits   = model(exp_imgs)
        pred_ids = logits.argmax(dim=1).cpu().numpy()

    # --- normalise shap_values into a helper that returns (C,H,W) for image i, class c ---
    def get_sv(i, c):
        sv = shap_values
        # Case A: list of arrays, one per class -> sv[c] shape (N,C,H,W)
        if isinstance(sv, list):
            arr = sv[c]                      # (N, C, H, W)
            return np.asarray(arr[i])
        # Case B: single ndarray
        arr = np.asarray(sv)
        # B1: (N, C, H, W, n_classes)  -> class dim last
        if arr.ndim == 5:
            return arr[i, :, :, :, c]
        # B2: (n_classes, N, C, H, W)  -> class dim first
        if arr.ndim == 5:
            return arr[c, i]
        # B3: (N, C, H, W) -> no explicit class dim (already class-specific)
        if arr.ndim == 4:
            return arr[i]
        raise ValueError(f"Unexpected SHAP output shape: {arr.shape}")

    shap_maps, shap_rgb = [], []
    for i in range(len(exp_imgs)):
        sv = get_sv(i, int(pred_ids[i]))        # (C, H, W)
        sv = np.asarray(sv)
        if sv.ndim == 2:                         # already (H,W)
            sv = sv[None, ...]

        unsigned = np.abs(sv).max(axis=0)
        p1, p99  = np.percentile(unsigned, 1), np.percentile(unsigned, 99)
        unsigned = np.clip(unsigned, p1, p99)
        unsigned = (unsigned - unsigned.min()) / (unsigned.max() - unsigned.min() + 1e-8)
        unsigned = gaussian_filter(unsigned, sigma=1)
        shap_maps.append(unsigned)

        signed  = sv.sum(axis=0)
        abs_max = np.abs(signed).max() + 1e-8
        signed  = signed / abs_max
        rgb     = np.zeros((*signed.shape, 3))
        rgb[..., 0] = np.clip( signed, 0, 1)
        rgb[..., 2] = np.clip(-signed, 0, 1)
        shap_rgb.append(rgb)

    return np.stack(shap_maps), np.stack(shap_rgb), pred_ids


# ═══════════════════════════════════════════════════════════════
# 5.  LIME — LimeImageExplainer
#     Segments image into superpixels, perturbs them, fits a
#     local linear model to rank region importance.
# ═══════════════════════════════════════════════════════════════
def compute_lime(model, exp_imgs, device, class_names,
                 num_samples=500, num_features=10):
    """
    Returns
    -------
    lime_boundaries : (N, H, W, 3) numpy  — segment outlines
    lime_masks      : (N, H, W)    numpy  — positive importance mask
    pred_ids        : (N,) int
    """
    model.eval()

    def predict_fn(images_np):
        """images_np: (B,H,W,3) float64 → (B, n_classes) probs"""
        batch = torch.tensor(
            images_np.transpose(0, 3, 1, 2),
            dtype=torch.float32
        ).to(device)
        with torch.no_grad():
            probs = F.softmax(model(batch), dim=1).cpu().numpy()
        return probs

    explainer = lime_image.LimeImageExplainer(random_state=42)
    imgs_np   = denorm(exp_imgs)

    with torch.no_grad():
        pred_ids = model(exp_imgs).argmax(dim=1).cpu().numpy()

    lime_boundaries = []
    lime_masks      = []

    for i in range(len(exp_imgs)):
        explanation = explainer.explain_instance(
            imgs_np[i].astype(np.float64),
            predict_fn,
            top_labels  = len(class_names),
            hide_color  = 0,
            num_samples = num_samples,
            random_seed = 42,
        )

        # Both positive and negative segments (full picture)
        _, mask = explanation.get_image_and_mask(
            pred_ids[i],
            positive_only = False,
            num_features  = num_features,
            hide_rest     = False,
        )
        boundary_img = mark_boundaries(
            imgs_np[i], mask,
            color=(1, 1, 0),
            outline_color=(0, 0, 0)
        )

        # Positive-only mask for clean overlay
        _, pos_mask = explanation.get_image_and_mask(
            pred_ids[i],
            positive_only = True,
            num_features  = num_features,
            hide_rest     = False,
        )

        lime_boundaries.append(boundary_img)
        lime_masks.append(pos_mask.astype(np.float32))
        print(f"  ✓ LIME image {i+1}/{len(exp_imgs)} done")

    return (np.stack(lime_boundaries),
            np.stack(lime_masks),
            pred_ids)


# ═══════════════════════════════════════════════════════════════
# 6.  FEATURE MAPS  (single image, named layer)
# ═══════════════════════════════════════════════════════════════
def get_feature_maps(model, single_input, layer_name="conv4"):
    features = {}

    def hook(module, inp, out):
        features[layer_name] = out.detach()

    handle = getattr(model, layer_name).register_forward_hook(hook)
    model.eval()
    with torch.no_grad():
        model(single_input)
    handle.remove()
    return features[layer_name].squeeze(0)


def plot_feature_maps(feature_maps, layer_name="conv4", n_maps=16):
    cols = 8
    n    = min(n_maps, feature_maps.shape[0])
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 1.9))
    axes = axes.flatten()

    for i in range(n):
        f = feature_maps[i].cpu().numpy()
        f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        axes[i].imshow(f, cmap="inferno")
        axes[i].set_title(f"ch {i}", fontsize=7)
        axes[i].axis("off")
    for j in range(n, len(axes)):
        axes[j].axis("off")

    fig.suptitle(f"Feature Maps — {layer_name}  (first {n} channels)",
                 fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"plot_I_feature_maps_{layer_name}.png",
                bbox_inches="tight", dpi=130)
    plt.show()
    print(f"  Saved → plot_I_feature_maps_{layer_name}.png")


# ═══════════════════════════════════════════════════════════════
# 7.  PLOT A — Grad-CAM Grid
#     Original | Grad-CAM Heatmap | Overlay
# ═══════════════════════════════════════════════════════════════
def plot_gradcam_grid(imgs_np, cams, pred_ids, labels, class_names):
    N = len(imgs_np)
    fig, axes = plt.subplots(N, 3, figsize=(11, 3.5 * N))
    if N == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original",
                                  "Grad-CAM Heatmap",
                                  "Overlay"]):
        axes[0, col].set_title(title, fontsize=12,
                               fontweight="bold", pad=8)

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        tick    = "✓" if correct else "✗"
        row_lbl = (f"{tick} True: {class_names[labels[i]]}\n"
                   f"   Pred: {class_names[pred_ids[i]]}")

        axes[i, 0].imshow(imgs_np[i])
        axes[i, 0].set_ylabel(row_lbl, fontsize=8,
                              rotation=0, labelpad=110, va="center")
        axes[i, 1].imshow(get_heatmap(cams[i], "jet"))
        axes[i, 2].imshow(get_overlay(imgs_np[i], cams[i]))

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle("Grad-CAM — CNN  |  Diabetic Retinopathy",
                 fontsize=14, y=1.001)
    plt.tight_layout()
    plt.savefig("plot_A_gradcam_grid.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("  Saved → plot_A_gradcam_grid.png")


# ═══════════════════════════════════════════════════════════════
# 8.  PLOT B — Correct vs Incorrect
# ═══════════════════════════════════════════════════════════════
def plot_correct_vs_wrong(imgs_np, cams, pred_ids, labels,
                          class_names, n=3):
    correct_idx   = [i for i in range(len(pred_ids))
                     if pred_ids[i] == labels[i]]
    incorrect_idx = [i for i in range(len(pred_ids))
                     if pred_ids[i] != labels[i]]

    c_idx  = correct_idx[:n]
    w_idx  = incorrect_idx[:n]
    n_rows = max(len(c_idx), len(w_idx))

    if n_rows == 0:
        print("  Plot B skipped — no examples for one category.")
        return

    fig, axes = plt.subplots(n_rows, 4, figsize=(14, 3.8 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for col, h in enumerate(["Correct — Original", "Correct — Overlay",
                              "Wrong — Original",   "Wrong — Overlay"]):
        axes[0, col].set_title(h, fontsize=10, fontweight="bold")

    for row in range(n_rows):
        for col_offset, pool in enumerate([c_idx, w_idx]):
            cb = col_offset * 2
            if row < len(pool):
                idx = pool[row]
                lbl = (f"T: {class_names[labels[idx]]}\n"
                       f"P: {class_names[pred_ids[idx]]}")
                axes[row, cb].imshow(imgs_np[idx])
                axes[row, cb].set_title(lbl, fontsize=8)
                axes[row, cb + 1].imshow(
                    get_overlay(imgs_np[idx], cams[idx]))
            for ax in axes[row, cb:cb + 2]:
                ax.axis("off")

    plt.suptitle("Grad-CAM — Correct vs Incorrect Predictions",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig("plot_B_correct_vs_wrong.png", bbox_inches="tight", dpi=140)
    plt.show()
    print("  Saved → plot_B_correct_vs_wrong.png")


# ═══════════════════════════════════════════════════════════════
# 9.  PLOT C — Mean CAM Intensity per predicted class
# ═══════════════════════════════════════════════════════════════
def plot_mean_cam_intensity(cams, pred_ids, class_names):
    buckets = {cn: [] for cn in class_names}
    for i, cam in enumerate(cams):
        buckets[class_names[pred_ids[i]]].append(cam.mean())

    means  = [np.mean(v) if v else 0.0 for v in buckets.values()]
    colors = [CLASS_COLOURS[i] for i in range(len(class_names))]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(class_names, means, color=colors,
                  edgecolor="black", linewidth=0.7)
    ax.bar_label(bars, fmt="%.4f", fontsize=9, padding=3)
    ax.set_xlabel("Predicted DR Class", fontsize=11)
    ax.set_ylabel("Mean CAM Activation", fontsize=11)
    ax.set_title("Mean Grad-CAM Activation per Predicted Class\n"
                 "(higher → model attends to a larger / more intense region)",
                 fontsize=11)
    ax.set_ylim(0, max(means) * 1.35 if max(means) > 0 else 1)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.savefig("plot_C_mean_intensity.png", bbox_inches="tight", dpi=130)
    plt.show()
    print("  Saved → plot_C_mean_intensity.png")


# ═══════════════════════════════════════════════════════════════
# 10. PLOT D — Softmax Confidence Grid
# ═══════════════════════════════════════════════════════════════
def plot_confidence_grid(logits, pred_ids, labels, class_names):
    N     = len(logits)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    cols  = 5
    rows  = (N + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols,
                             figsize=(cols * 3.2, rows * 3))
    axes = axes.flatten()

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        colours = ["#4CAF50" if j == pred_ids[i] else
                   "#F44336" if j == labels[i] and not correct else
                   "#90CAF9"
                   for j in range(len(class_names))]
        axes[i].bar(class_names, probs[i], color=colours,
                    edgecolor="black", linewidth=0.5)
        axes[i].set_title(
            f"{'✓' if correct else '✗'}  "
            f"T:{class_names[labels[i]]}\n"
            f"P:{class_names[pred_ids[i]]} "
            f"({probs[i][pred_ids[i]]:.0%})",
            fontsize=8
        )
        axes[i].set_ylim(0, 1)
        axes[i].set_xticks(range(len(class_names)))
        axes[i].set_xticklabels(class_names, rotation=45,
                                ha="right", fontsize=7)
        axes[i].set_ylabel("Prob", fontsize=7)
        axes[i].spines["top"].set_visible(False)
        axes[i].spines["right"].set_visible(False)

    for j in range(N, len(axes)):
        axes[j].axis("off")

    legend_handles = [
        Patch(color="#4CAF50", label="Predicted class"),
        Patch(color="#F44336", label="True class (wrong pred)"),
        Patch(color="#90CAF9", label="Other classes"),
    ]
    fig.legend(handles=legend_handles, loc="lower center",
               ncol=3, bbox_to_anchor=(0.5, -0.03), fontsize=9)
    plt.suptitle("CNN — Softmax Confidence per Image", fontsize=13)
    plt.tight_layout()
    plt.savefig("plot_D_confidence_grid.png", bbox_inches="tight", dpi=130)
    plt.show()
    print("  Saved → plot_D_confidence_grid.png")


# ═══════════════════════════════════════════════════════════════
# 11. PLOT E — Saliency Map Grid
#     Original | Saliency Map | Saliency Overlay
# ═══════════════════════════════════════════════════════════════
def plot_saliency_grid(imgs_np, saliencies, pred_ids,
                       labels, class_names):
    N = len(imgs_np)
    fig, axes = plt.subplots(N, 3, figsize=(11, 3.5 * N))
    if N == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original",
                                  "Saliency Map",
                                  "Saliency Overlay"]):
        axes[0, col].set_title(title, fontsize=12,
                               fontweight="bold", pad=8)

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        tick    = "✓" if correct else "✗"
        row_lbl = (f"{tick} True: {class_names[labels[i]]}\n"
                   f"   Pred: {class_names[pred_ids[i]]}")

        axes[i, 0].imshow(imgs_np[i])
        axes[i, 0].set_ylabel(row_lbl, fontsize=8,
                              rotation=0, labelpad=110, va="center")
        axes[i, 1].imshow(get_heatmap(saliencies[i], "inferno"))
        axes[i, 2].imshow(get_overlay(imgs_np[i], saliencies[i],
                                      colormap="inferno", alpha=0.55))

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle("Saliency Maps — CNN  |  Diabetic Retinopathy",
                 fontsize=14, y=1.001)
    plt.tight_layout()
    plt.savefig("plot_E_saliency_grid.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("  Saved → plot_E_saliency_grid.png")


# ═══════════════════════════════════════════════════════════════
# 12. PLOT F — SHAP Grid
#     Original | SHAP Intensity | SHAP Signed | SHAP Overlay
# ═══════════════════════════════════════════════════════════════
def plot_shap_grid(imgs_np, shap_maps, shap_rgb,
                   pred_ids, labels, class_names):
    N = len(imgs_np)
    fig, axes = plt.subplots(N, 4, figsize=(15, 3.5 * N))
    if N == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original",
                                  "SHAP Intensity",
                                  "SHAP Signed (R=+, B=−)",
                                  "SHAP Overlay"]):
        axes[0, col].set_title(title, fontsize=11,
                               fontweight="bold", pad=8)

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        tick    = "✓" if correct else "✗"
        row_lbl = (f"{tick} True: {class_names[labels[i]]}\n"
                   f"   Pred: {class_names[pred_ids[i]]}")

        axes[i, 0].imshow(imgs_np[i])
        axes[i, 0].set_ylabel(row_lbl, fontsize=8,
                              rotation=0, labelpad=110, va="center")
        axes[i, 1].imshow(get_heatmap(shap_maps[i], "jet"))
        axes[i, 2].imshow(np.clip(shap_rgb[i], 0, 1))
        axes[i, 3].imshow(get_overlay(imgs_np[i], shap_maps[i]))

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle("SHAP GradientExplainer — CNN  |  Diabetic Retinopathy",
                 fontsize=14, y=1.001)
    plt.tight_layout()
    plt.savefig("plot_F_shap_grid.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("  Saved → plot_F_shap_grid.png")


# ═══════════════════════════════════════════════════════════════
# 13. PLOT G — LIME Grid
#     Original | LIME Segments | Positive Mask Overlay
# ═══════════════════════════════════════════════════════════════
def plot_lime_grid(imgs_np, lime_boundaries, lime_masks,
                   pred_ids, labels, class_names):
    N = len(imgs_np)
    fig, axes = plt.subplots(N, 3, figsize=(11, 3.5 * N))
    if N == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original",
                                  "LIME Segments",
                                  "Positive Mask Overlay"]):
        axes[0, col].set_title(title, fontsize=12,
                               fontweight="bold", pad=8)

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        tick    = "✓" if correct else "✗"
        row_lbl = (f"{tick} True: {class_names[labels[i]]}\n"
                   f"   Pred: {class_names[pred_ids[i]]}")

        mask_norm = lime_masks[i]
        if mask_norm.max() > 0:
            mask_norm = mask_norm / mask_norm.max()

        axes[i, 0].imshow(imgs_np[i])
        axes[i, 0].set_ylabel(row_lbl, fontsize=8,
                              rotation=0, labelpad=110, va="center")
        axes[i, 1].imshow(np.clip(lime_boundaries[i], 0, 1))
        axes[i, 2].imshow(get_overlay(imgs_np[i], mask_norm,
                                      colormap="Greens", alpha=0.5))

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle("LIME Explainer — CNN  |  Diabetic Retinopathy",
                 fontsize=14, y=1.001)
    plt.tight_layout()
    plt.savefig("plot_G_lime_grid.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("  Saved → plot_G_lime_grid.png")


# ═══════════════════════════════════════════════════════════════
# 14. PLOT H — Full Method Comparison  ← CORRECTED
#     Original | Grad-CAM | Saliency | SHAP | LIME
# ═══════════════════════════════════════════════════════════════
def plot_method_comparison(imgs_np, cams, saliencies,
                           shap_maps, lime_boundaries,
                           pred_ids, labels, class_names):
    """
    5-column grid comparing all four methods on the same image.
    Where methods agree → high-confidence diagnostic region.
    Where they disagree → worth investigating further.
    """
    N = len(imgs_np)
    fig, axes = plt.subplots(N, 5, figsize=(19, 3.5 * N))
    if N == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(["Original",
                                  "Grad-CAM Overlay",
                                  "Saliency Map",
                                  "SHAP Intensity",
                                  "LIME Segments"]):
        axes[0, col].set_title(title, fontsize=11,
                               fontweight="bold", pad=8)

    for i in range(N):
        correct = pred_ids[i] == labels[i]
        tick    = "✓" if correct else "✗"
        row_lbl = (f"{tick} True: {class_names[labels[i]]}\n"
                   f"   Pred: {class_names[pred_ids[i]]}")

        axes[i, 0].imshow(imgs_np[i])
        axes[i, 0].set_ylabel(row_lbl, fontsize=8,
                              rotation=0, labelpad=110, va="center")
        axes[i, 1].imshow(get_overlay(imgs_np[i], cams[i]))
        axes[i, 2].imshow(get_heatmap(saliencies[i], "inferno"))
        axes[i, 3].imshow(get_heatmap(shap_maps[i], "jet"))
        axes[i, 4].imshow(np.clip(lime_boundaries[i], 0, 1))

        for ax in axes[i]:
            ax.axis("off")

    plt.suptitle(
        "Explainability Comparison — Grad-CAM vs Saliency vs SHAP vs LIME\n"
        "CNN  |  Diabetic Retinopathy",
        fontsize=13, y=1.001
    )
    plt.tight_layout()
    plt.savefig("plot_H_method_comparison.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("  Saved → plot_H_method_comparison.png")


# ═══════════════════════════════════════════════════════════════
# 15. MAIN — run full pipeline
# ═══════════════════════════════════════════════════════════════
def run_full_explainability(
    model,
    dataset,
    device,
    n_samples      = 10,
    gradcam_layer  = "conv4",
    featmap_layer  = "conv4",
    n_feature_maps = 16,
    n_background   = 50,     # SHAP background images
    lime_samples   = 500,    # LIME perturbations (↑ = better, slower)
    lime_features  = 10,     # top superpixel segments to highlight
    seed           = 42,
):
    class_names = dataset.classes
    print(f"Classes  : {class_names}")
    print(f"Samples  : {n_samples}  |  Grad-CAM layer : {gradcam_layer}")
    print(f"LIME perturbations : {lime_samples}\n")

    # ── Sample ──────────────────────────────────────────────────
    imgs, labels, _ = sample_dataset(dataset, n_samples, seed=seed)
    imgs    = imgs.to(device)
    imgs_np = denorm(imgs)

    # ── Grad-CAM ────────────────────────────────────────────────
    print("→ Running Grad-CAM …")
    target_layer           = getattr(model, gradcam_layer)
    gcam                   = GradCAM(model, target_layer)
    cams, pred_ids, logits = gcam.generate(imgs)
    print(f"  ✓ Grad-CAM done   — shape : {cams.shape}")

    # ── Saliency ────────────────────────────────────────────────
    print("→ Running Saliency …")
    saliencies = compute_saliency_batch(model, imgs.clone().detach())
    print(f"  ✓ Saliency done   — shape : {saliencies.shape}")

    # ── SHAP ────────────────────────────────────────────────────
    print("→ Running SHAP GradientExplainer …")
    shap_maps, shap_rgb, shap_pids = compute_shap(
        model, imgs, device, dataset,
        n_background=n_background, seed=seed
    )
    print(f"  ✓ SHAP done       — shape : {shap_maps.shape}")

    # ── LIME ────────────────────────────────────────────────────
    print("→ Running LIME (~1–2 min for 10 images) …")
    lime_boundaries, lime_masks, lime_pids = compute_lime(
        model, imgs, device, class_names,
        num_samples=lime_samples,
        num_features=lime_features
    )
    print(f"  ✓ LIME done       — shape : {lime_boundaries.shape}")

    # ── Plot A : Grad-CAM grid ───────────────────────────────────
    print("\n→ Plot A : Grad-CAM grid …")
    plot_gradcam_grid(imgs_np, cams, pred_ids, labels, class_names)

    # ── Plot B : Correct vs Wrong ────────────────────────────────
    print("→ Plot B : Correct vs Incorrect …")
    plot_correct_vs_wrong(imgs_np, cams, pred_ids,
                          labels, class_names, n=3)

    # ── Plot C : Mean CAM intensity ──────────────────────────────
    print("→ Plot C : Mean CAM intensity per class …")
    plot_mean_cam_intensity(cams, pred_ids, class_names)

    # ── Plot D : Confidence grid ─────────────────────────────────
    print("→ Plot D : Softmax confidence grid …")
    plot_confidence_grid(logits, pred_ids, labels, class_names)

    # ── Plot E : Saliency grid ───────────────────────────────────
    print("→ Plot E : Saliency map grid …")
    plot_saliency_grid(imgs_np, saliencies, pred_ids,
                       labels, class_names)

    # ── Plot F : SHAP grid ───────────────────────────────────────
    print("→ Plot F : SHAP grid …")
    plot_shap_grid(imgs_np, shap_maps, shap_rgb,
                   shap_pids, labels, class_names)

    # ── Plot G : LIME grid ───────────────────────────────────────
    print("→ Plot G : LIME grid …")
    plot_lime_grid(imgs_np, lime_boundaries, lime_masks,
                   lime_pids, labels, class_names)

    # ── Plot H : Full method comparison ─────────────────────────
    print("→ Plot H : Full method comparison …")
    plot_method_comparison(imgs_np, cams, saliencies,
                           shap_maps, lime_boundaries,
                           pred_ids, labels, class_names)

    # ── Plot I : Feature maps ────────────────────────────────────
    print(f"→ Plot I : Feature maps — layer '{featmap_layer}' …")
    fmaps = get_feature_maps(model, imgs[0:1], layer_name=featmap_layer)
    plot_feature_maps(fmaps, layer_name=featmap_layer,
                      n_maps=n_feature_maps)

    print("\n✅  Full explainability pipeline complete.")
    print("\n   Output files:")
    for letter, name in zip("ABCDEFGHI", [
        "Grad-CAM Grid",
        "Correct vs Wrong",
        "Mean CAM Intensity",
        "Confidence Grid",
        "Saliency Grid",
        "SHAP Grid",
        "LIME Grid",
        "Method Comparison",
        "Feature Maps",
    ]):
        print(f"   plot_{letter}_*.png  →  {name}")


## Run the four methods on the frozen 30-image set (upsampled to 299)

In [6]:
import torch.nn.functional as F
XAISET='/kaggle/input/notebooks/tochyokafor/06-build-xai-imageset'
xai_imgs=np.load(f'{XAISET}/xai_images.npy')      # (30,3,224,224)
xai_labels=np.load(f'{XAISET}/xai_labels.npy')
class_names=open(f'{XAISET}/xai_class_names.txt').read().splitlines()
imgs=torch.tensor(xai_imgs,dtype=torch.float32)
imgs=F.interpolate(imgs,size=(299,299),mode='bilinear',align_corners=False).to(device)  # 224->299
labels=list(xai_labels)
imgs_np=denorm(imgs)
print('Frozen set upsampled to:', imgs.shape)

print('Grad-CAM ...')
gcam=GradCAM(inception, inception.Mixed_7c)   # deepest inception block
cams,pred_ids,logits=gcam.generate(imgs)
print('Saliency ...')
saliencies=compute_saliency_batch(inception, imgs.clone().detach())
print('SHAP ...')
shap_maps,shap_rgb,shap_pids=compute_shap(inception, imgs, device, dataset, n_background=50)
print('LIME ...')
lime_boundaries,lime_masks,lime_pids=compute_lime(inception, imgs, device, class_names, num_samples=500)
print('Done. Heatmaps: cams, saliencies, shap_maps, lime_masks')

Frozen set upsampled to: torch.Size([30, 3, 299, 299])
Grad-CAM ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Saliency ...
SHAP ...
LIME ...


  0%|          | 0/500 [00:00<?, ?it/s]

  ✓ LIME image 1/30 done


...

  ✓ LIME image 28/30 done


  0%|          | 0/500 [00:00<?, ?it/s]

  ✓ LIME image 29/30 done


  0%|          | 0/500 [00:00<?, ?it/s]

  ✓ LIME image 30/30 done
Done. Heatmaps: cams, saliencies, shap_maps, lime_masks


## Block 2 - deletion/insertion (blurred baseline) + save heatmaps

In [7]:
from scipy.ndimage import gaussian_filter
import torch.nn.functional as F, json
MODEL_NAME = "inception_v3"; OUT = "/kaggle/working"
METHODS = {"gradcam":cams, "saliency":saliencies, "shap":shap_maps, "lime":lime_masks}

def blurred_baseline(img_t, sigma=12):
    a=img_t.detach().cpu().numpy()
    for c in range(a.shape[0]): a[c]=gaussian_filter(a[c],sigma=sigma)
    return torch.tensor(a,dtype=img_t.dtype,device=img_t.device)

@torch.no_grad()
def prob_of(m,img,cls):
    o=m(img.unsqueeze(0)); o=o.logits if hasattr(o,"logits") else o
    return torch.softmax(o,1)[0,cls].item()

def del_ins(m,img,heat,cls,steps=50):
    H,W=heat.shape; order=np.argsort(heat.ravel())[::-1]; blur=blurred_baseline(img)
    d=img.clone(); dp=[prob_of(m,d,cls)]; ins=blur.clone(); ip=[prob_of(m,ins,cls)]
    ch=max(1,len(order)//steps)
    for s in range(0,len(order),ch):
        px=order[s:s+ch]; ys,xs=np.unravel_index(px,(H,W))
        d[:,ys,xs]=blur[:,ys,xs]; ins[:,ys,xs]=img[:,ys,xs]
        dp.append(prob_of(m,d,cls)); ip.append(prob_of(m,ins,cls))
    return np.trapz(dp)/len(dp), np.trapz(ip)/len(ip)

res={m:{"deletion":[],"insertion":[]} for m in METHODS}
preds=model(imgs).argmax(1).cpu().numpy()
for mth,maps in METHODS.items():
    for i in range(len(imgs)):
        heat=np.asarray(maps[i],dtype=float)
        if heat.shape!=tuple(imgs.shape[2:]):
            heat=F.interpolate(torch.tensor(heat)[None,None],size=tuple(imgs.shape[2:]),
                               mode="bilinear",align_corners=False)[0,0].numpy()
        heat=(heat-heat.min())/(np.ptp(heat)+1e-8)
        d,ii=del_ins(model,imgs[i],heat,int(preds[i])); res[mth]["deletion"].append(d); res[mth]["insertion"].append(ii)
    np.save(f"{OUT}/{MODEL_NAME}_{mth}_heatmaps.npy",
            np.stack([(lambda h:(h-h.min())/(np.ptp(h)+1e-8))(np.asarray(maps[i],dtype=float)) for i in range(len(maps))]))

print(f"\n=== {MODEL_NAME}: deletion (lower=better) / insertion (higher=better) ===")
for m in METHODS:
    print(f"{m:<10}{np.mean(res[m]['deletion']):<12.4f}{np.mean(res[m]['insertion']):.4f}")
json.dump({m:{"deletion":float(np.mean(res[m]["deletion"])),"insertion":float(np.mean(res[m]["insertion"]))} for m in METHODS},
          open(f"{OUT}/{MODEL_NAME}_delins.json","w"), indent=2)
print("Saved", MODEL_NAME, "heatmaps + delins.json")

/tmp/ipykernel_23/682969283.py:24: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(dp)/len(dp), np.trapz(ip)/len(ip)



=== inception_v3: deletion (lower=better) / insertion (higher=better) ===
gradcam   0.8168      0.8191
saliency  0.8239      0.8521
shap      0.8202      0.8344
lime      0.8405      0.8305
Saved inception_v3 heatmaps + delins.json
